# 00 Dataset Overview

Dataset summary, class counts, marker distributions, and split sanity checks.

In [ ]:
from pathlib import Path
import os
import sys
import json

def _find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for path in candidates:
        if (path / 'pyproject.toml').exists() and (path / 'src' / 'cytof_archetypes').exists():
            return path
    fallback = Path('/Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv')
    if (fallback / 'src' / 'cytof_archetypes').exists():
        return fallback
    raise RuntimeError('Could not locate repository root containing src/cytof_archetypes')

REPO_ROOT = _find_repo_root()
SRC_DIR = REPO_ROOT / 'src'
def _resolve_out_dir() -> Path:
    env = os.environ.get('CYTOF_SUITE_OUTPUT_DIR')
    if env:
        return Path(env)
    cfg_path = REPO_ROOT / 'configs' / 'experiment_suite.yaml'
    if cfg_path.exists():
        try:
            import yaml
            cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8')) or {}
            out_raw = cfg.get('output_dir')
            if out_raw:
                out_path = Path(out_raw)
                return out_path if out_path.is_absolute() else REPO_ROOT / out_path
        except Exception:
            pass
    return REPO_ROOT / 'outputs' / 'experiment_suite'

OUT_DIR = _resolve_out_dir()

def _artifact_exists(path: Path) -> bool:
    if path.exists():
        return True
    print(f'Missing artifact: {path}')
    print('Run the suite first: python scripts/run_experiment_suite.py --config configs/experiment_suite.yaml')
    return False
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print('Repo root:', REPO_ROOT)
print('Using src dir:', SRC_DIR)
print('Using suite output dir:', OUT_DIR)


In [ ]:
from pathlib import Path
import pandas as pd
from cytof_archetypes.datasets import load_dataset_bundle
dataset_cfg = {
    'name': 'levine32',
    'input_path': str(REPO_ROOT / 'data' / 'levine32_processed.h5ad'),
    'label_column': 'label',
    'cell_id_column': 'cell_id',
    'val_fraction': 0.15,
    'test_fraction': 0.15,
}
bundle = load_dataset_bundle(dataset_cfg, seed=42)
print('markers:', len(bundle.markers))
print('train/val/test sizes:', len(bundle.train.x), len(bundle.val.x), len(bundle.test.x))

In [ ]:
if bundle.train.labels is not None:
    print('train label counts')
    print(pd.Series(bundle.train.labels).value_counts())
if bundle.val.labels is not None:
    print('val label counts')
    print(pd.Series(bundle.val.labels).value_counts())

In [ ]:
split_manifest = OUT_DIR / 'reports' / 'split_manifest.csv'
if split_manifest.exists():
    manifest = pd.read_csv(split_manifest)
    display(manifest.head())
    print(manifest['split'].value_counts())